# Search Query Agent - SFT Training

Works on **Google Colab**, **NVIDIA Cloud**, or any Jupyter with a GPU.

In [ ]:
# [1] Install dependencies (latest transformers required for Mistral3 model support)
!pip install -q torch datasets accelerate peft bitsandbytes sentencepiece protobuf scipy numpy wandb
!pip install -q trl
!pip install -q git+https://github.com/huggingface/transformers.git

In [ ]:
# [2] Check GPU + Login to HuggingFace + W&B
import torch, os
print(f"GPU: {torch.cuda.is_available()} - {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > GPU")

from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))  # or: huggingface-cli login

import wandb
wandb.login(key=os.environ.get("WANDB_API_KEY"))  # or: wandb login
print("W&B logged in!")

In [ ]:
# [3] Upload your files
# Upload these 3 files into a folder called "search_query_agent/" next to this notebook:
#   1. train.py
#   2. system_prompt.md
#   3. data.jsonl
#
# NVIDIA Cloud / Jupyter: Use the file browser on the left sidebar to upload.
# Google Colab: Uncomment the colab block below instead.

import os

os.makedirs("search_query_agent", exist_ok=True)

# --- Option A: For NVIDIA Cloud / standard Jupyter ---
# Upload via the file browser sidebar into the "search_query_agent" folder.
# Then just run this cell to verify.

# --- Option B: For Google Colab (uncomment these lines) ---
# from google.colab import files
# print("Select train.py, system_prompt.md, and data.jsonl:")
# uploaded = files.upload()
# for name, content in uploaded.items():
#     with open(f"search_query_agent/{name}", "wb") as f:
#         f.write(content)

# Verify files exist
required = ["train.py", "system_prompt.md", "data.jsonl"]
found = os.listdir("search_query_agent") if os.path.isdir("search_query_agent") else []
print("Files in search_query_agent/:")
for f in sorted(found):
    print(f"  {f}")

missing = [r for r in required if r not in found]
if missing:
    print(f"\nMISSING: {missing}")
    print("Upload them into the 'search_query_agent' folder and re-run this cell.")
else:
    print("\nAll 3 files found. Ready to train!")

### Fine-tuning parameters (used for this model)

| Parameter | Value | Note |
|-----------|--------|------|
| **Model** | `mistralai/Ministral-3-14B-Instruct-2512-BF16` | Base model |
| **Epochs** | 3 | |
| **Learning rate** | 1e-4 | Overrides train.py default 2e-4 |
| **LoRA r** | 16 | |
| **LoRA alpha** | 32 | rsLoRA scaling = alpha/√r |
| **Batch size** | 4 (train.py default) | |
| **Gradient accumulation** | 4 (train.py default) | Effective batch = 16 |
| **Max sequence length** | 2048 (train.py default) | |
| **Inference (this model)** | max_new_tokens=80, temperature=0 | Same as Mistral API eval for comparable results |

In [ ]:
# [4] Run training with Ministral-14B
# Eval graphs: train loss & eval loss curves appear in W&B (project: search-query-agent).
# Add the line below only when you want to upload to HuggingFace after this run:
#     --hub_repo livawork/search-query-agent-ministral-14b
!python search_query_agent/train.py \
    --model_name mistralai/Ministral-3-14B-Instruct-2512-BF16 \
    --data_path search_query_agent/data.jsonl \
    --system_prompt_path search_query_agent/system_prompt.md \
    --epochs 3 \
    --lr 1e-4 \
    --lora_r 16 \
    --lora_alpha 32

In [ ]:
# [5] Test the trained model
from transformers import AutoTokenizer, AutoModelForImageTextToText, MistralForCausalLM
from peft import PeftModel
import torch, json, os

base_model_name = "mistralai/Ministral-3-14B-Instruct-2512-BF16"
adapter_path = os.path.abspath("./search-query-agent-sft")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

# Load base model (extract CausalLM from VLM)
print("Loading base model...")
full_model = AutoModelForImageTextToText.from_pretrained(
    base_model_name, trust_remote_code=True, torch_dtype=torch.bfloat16, device_map="auto"
)
model = MistralForCausalLM.__new__(MistralForCausalLM)
torch.nn.Module.__init__(model)
text_config = full_model.config.text_config
text_config.tie_word_embeddings = False
model.config = text_config
model.model = full_model.model.language_model
model.lm_head = full_model.lm_head
model.generation_config = full_model.generation_config
del full_model

# Load LoRA adapter
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, adapter_path)
model = model.merge_and_unload()
print("Model ready!")

system_prompt = open("search_query_agent/system_prompt.md").read().strip()

def generate_query(question):
    user_input = json.dumps({"user_input": question})
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    # Same inference params as Mistral API: 80 tokens, temp=0 (greedy)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            stop_strings=["}"],
            tokenizer=tokenizer,
        )
    raw = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if not raw.endswith("}"):
        raw = raw + '"}'
    return raw

test_questions = [
    "Which of the following is the primary function of hemoglobin? A) Transport oxygen B) Fight infection C) Clot blood D) Digest food",
    "What is the time complexity of binary search? A) O(n) B) O(log n) C) O(n^2) D) O(1)",
    "The Treaty of Westphalia (1648) is most associated with: A) End of WWI B) Modern state sovereignty C) Colonial expansion D) Religious reform",
    "Which organelle is responsible for ATP production in eukaryotic cells? A) Nucleus B) Ribosome C) Mitochondria D) Golgi apparatus",
    "What is the primary cause of ocean acidification? A) Volcanic eruptions B) CO2 absorption C) Oil spills D) Thermal pollution",
]

print("=" * 60)
print("MODEL TEST RESULTS")
print("=" * 60)
for i, q in enumerate(test_questions, 1):
    result = generate_query(q)
    print(f"\n[{i}] Q: {q[:80]}...")
    print(f"    A: {result}")

In [ ]:
# Inference testing — Mistral API only (no local model)
# Set MISTRAL_API_KEY in env or below. Uses system prompt + your data, calls API, compares to expected.
# Logs eval metrics to W&B for graphs.
import json
import os
import requests
import wandb

MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY", "YOUR_MISTRAL_API_KEY")  # set your key
MISTRAL_API_URL = "https://api.mistral.ai/v1/chat/completions"
MODEL = "open-mistral-7b"  # or "mistral-small-latest", "codestral-latest" etc.
DATA_PATH = "search_query_agent/data.jsonl"
SYSTEM_PROMPT_PATH = "search_query_agent/system_prompt.md"
MAX_TEST = 10  # examples to test (set None for all)

with open(SYSTEM_PROMPT_PATH, "r", encoding="utf-8") as f:
    system_prompt = f.read().strip()

def load_pairs(path, max_examples=None):
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                r = json.loads(line)
                inp = r.get("input") or r.get("user_prompt", "")
                out = r.get("output", "")
                if isinstance(out, dict) and "query" in out:
                    out = json.dumps(out)
                pairs.append({"input": inp, "expected": out})
            except Exception:
                continue
            if max_examples and len(pairs) >= max_examples:
                break
    return pairs

def call_mistral_api(question):
    user_content = json.dumps({"user_input": question})
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
        "max_tokens": 80,
        "temperature": 0,
    }
    r = requests.post(
        MISTRAL_API_URL,
        headers={"Authorization": f"Bearer {MISTRAL_API_KEY}", "Content-Type": "application/json"},
        json=payload,
        timeout=60,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()

pairs = load_pairs(DATA_PATH, MAX_TEST)
print(f"Inference testing via Mistral API on {len(pairs)} examples from {DATA_PATH}\n")
print("=" * 70)

# W&B: log eval metrics for graphs
run = wandb.init(project="search-query-agent", job_type="inference_eval", name="inference-eval-api")
wandb.define_metric("inference_eval/example_idx")
wandb.define_metric("inference_eval/exact_match_per_example", step_metric="inference_eval/example_idx")
wandb.define_metric("inference_eval/exact_match_rate", step_metric="inference_eval/example_idx")

matches = 0
table_rows = []
for i, row in enumerate(pairs, 1):
    question = row["input"]
    expected = row["expected"]
    predicted = call_mistral_api(question)
    match = 1 if predicted.strip() == expected.strip() else 0
    matches += match
    status = "OK" if match else "DIFF"
    print(f"\n[{i}] {status}")
    print(f"  Q: {question[:100]}...")
    print(f"  Expected:  {expected}")
    print(f"  API:       {predicted}")
    # Log per-example for eval graphs
    wandb.log({
        "inference_eval/example_idx": i,
        "inference_eval/exact_match_per_example": match,
        "inference_eval/exact_match_rate": matches / i,
    })
    table_rows.append([str(i), question[:80] + ("..." if len(question) > 80 else ""), expected[:60], predicted[:60], "✓" if match else "✗"])

# Summary metrics and table
wandb.log({
    "inference_eval/exact_match_rate_final": matches / len(pairs) if pairs else 0,
    "inference_eval/total_correct": matches,
    "inference_eval/total_examples": len(pairs),
    "inference_eval/predictions": wandb.Table(columns=["#", "question", "expected", "predicted", "match"], data=table_rows),
})
run.finish()

print("\n" + "=" * 70)
print(f"Exact match: {matches}/{len(pairs)}")

In [ ]:
# [5b] Test on your data (expected vs model output)
# Run Cell 5 first so model and generate_query exist.
import json

DATA_PATH = "search_query_agent/data.jsonl"
MAX_TEST = 10  # number of examples from data to test (or None for all)

def load_input_output_pairs(path, max_examples=None):
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
                inp = record.get("input") or record.get("user_prompt", "")
                out = record.get("output", "")
                if isinstance(out, dict) and "query" in out:
                    out = json.dumps(out)
                pairs.append({"input": inp, "expected": out})
            except Exception:
                continue
            if max_examples and len(pairs) >= max_examples:
                break
    return pairs

pairs = load_input_output_pairs(DATA_PATH, MAX_TEST)
print(f"Testing on {len(pairs)} examples from {DATA_PATH}\n")
print("=" * 70)

for i, row in enumerate(pairs, 1):
    question = row["input"]
    expected = row["expected"]
    predicted = generate_query(question)
    match = "OK" if predicted.strip() == expected.strip() else "DIFF"
    print(f"\n[{i}] {match}")
    print(f"  Q: {question[:100]}...")
    print(f"  Expected:  {expected}")
    print(f"  Model:     {predicted}")

# Summary
matches = sum(1 for i, row in enumerate(pairs) if generate_query(row["input"]).strip() == row["expected"].strip())
print("\n" + "=" * 70)
print(f"Exact match: {matches}/{len(pairs)}")

In [ ]:
# [6] Inference eval: compare base vs fine-tuned loss on val set, log curves to W&B
import os, json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForImageTextToText, MistralForCausalLM
from peft import PeftModel
import wandb

BASE_MODEL = "mistralai/Ministral-3-14B-Instruct-2512-BF16"
ADAPTER_PATH = os.path.abspath("./search-query-agent-sft")
DATA_PATH = "search_query_agent/data.jsonl"
SYSTEM_PROMPT_PATH = "search_query_agent/system_prompt.md"
VAL_SPLIT = 0.1
SEED = 42
BATCH_SIZE = 4
MAX_LEN = 2048

# Load system prompt
with open(SYSTEM_PROMPT_PATH, "r", encoding="utf-8") as f:
    system_prompt = f.read().strip()

# Load and split data (same as train.py)
records = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
            inp = record.get("input") or record.get("user_prompt", "")
            out = record.get("output", "")
            if isinstance(out, dict) and "query" in out:
                out = json.dumps(out)
            records.append({"input": inp, "output": out})
        except Exception:
            continue
dataset = Dataset.from_list(records)
split = dataset.train_test_split(test_size=VAL_SPLIT, seed=SEED)
val_data = split["test"]

def to_messages(ex):
    user_content = json.dumps({"user_input": ex["input"]})
    return {"messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_content}, {"role": "assistant", "content": ex["output"]}]}

val_data = val_data.map(lambda x: to_messages(x))
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def apply_template(ex):
    text = tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

val_data = val_data.map(apply_template)

# Tokenize val set
def tokenize(ex):
    t = tokenizer(ex["text"], truncation=True, max_length=MAX_LEN, padding=False)
    t["labels"] = t["input_ids"].copy()
    return t

val_data = val_data.map(tokenize, remove_columns=val_data.column_names)

# Batch and pad (labels: -100 for padding so loss ignores those positions)
def collate(batch):
    ids = [b["input_ids"] for b in batch]
    labels = [b["labels"] for b in batch]
    padded = tokenizer.pad({"input_ids": ids}, return_tensors="pt", padding=True, return_attention_mask=True)
    labels_padded = torch.full_like(padded["input_ids"], -100, dtype=torch.long)
    for i, lab in enumerate(labels):
        labels_padded[i, :len(lab)] = torch.tensor(lab, dtype=torch.long)
    return {
        "input_ids": padded["input_ids"].to("cuda"),
        "attention_mask": padded["attention_mask"].to("cuda"),
        "labels": labels_padded.to("cuda"),
    }

# Load base model (CausalLM from VLM)
print("Loading base model...")
full = AutoModelForImageTextToText.from_pretrained(BASE_MODEL, trust_remote_code=True, torch_dtype=torch.bfloat16, device_map="auto")
base_model = MistralForCausalLM.__new__(MistralForCausalLM)
torch.nn.Module.__init__(base_model)
base_model.config = full.config.text_config
base_model.model = full.model.language_model
base_model.lm_head = full.lm_head
del full
torch.cuda.empty_cache()

# Load fine-tuned (base + adapter)
print("Loading fine-tuned adapter...")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

base_model.eval()
ft_model.eval()

# Compute loss per batch for both models -> curves
losses_base, losses_ft = [], []
n = len(val_data)
for start in range(0, n, BATCH_SIZE):
    batch = [val_data[i] for i in range(start, min(start + BATCH_SIZE, n))]
    batch = collate(batch)
    with torch.no_grad():
        out_base = base_model(**batch)
        out_ft = ft_model(**batch)
    losses_base.append(out_base.loss.item())
    losses_ft.append(out_ft.loss.item())

# Log to W&B (loss curve = per-batch loss)
wandb.init(project="search-query-agent", name="inference-eval-compare", tags=["inference-eval"])
wandb.define_metric("inference_eval/step")
wandb.define_metric("inference_eval/*", step_metric="inference_eval/step")
for step, (lb, lf) in enumerate(zip(losses_base, losses_ft)):
    wandb.log({"inference_eval/step": step, "inference_eval/loss_base": lb, "inference_eval/loss_finetuned": lf})
wandb.log({
    "inference_eval/mean_loss_base": sum(losses_base) / len(losses_base),
    "inference_eval/mean_loss_finetuned": sum(losses_ft) / len(losses_ft),
})
wandb.finish()
print("Done. Check W&B for inference_eval/loss_base and inference_eval/loss_finetuned curves.")

In [ ]:
# [6] (Optional) Run with evaluation
# !python search_query_agent/train.py \
#     --model_name mistralai/Ministral-3-14B-Instruct-2512-BF16 \
#     --data_path search_query_agent/data.jsonl \
#     --system_prompt_path search_query_agent/system_prompt.md \
#     --no_wandb

In [ ]:
# [7] Download LoRA adapter
!tar -czf adapter.tar.gz ./search-query-agent-sft
print("Done! Download 'adapter.tar.gz' from the file browser sidebar on the left.")